# 00 · Prepare the images on Google Colab (run once)

Use this notebook instead of `01_data_exploration.ipynb` when you train on Google Colab and don't want the 42 GB dataset on your own computer.

**What it does:**
1. Downloads the 12 NIH ChestX-ray14 archives, one at a time, onto the Colab machine's temporary disk.
2. Keeps only the 44,828 X-rays in this project's splits (`data/splits/`), so the train, validation and test patients stay exactly as before. It shrinks them from 1024×1024 to 512×512.
3. Packs them into one file, `images_512.tar` (about 5 GB), in your Google Drive at `MyDrive/chest_xray_classifier/`.

`02_train.ipynb` and `03_evaluate.ipynb` then unpack this file at the start of each Colab session, which takes a few minutes, instead of downloading everything again.

**Requirements:**
- About 5 GB of free space in Google Drive.
- Roughly an hour, depending on download speed. Keep the browser tab open while it runs.

**Why 512×512:** it is small enough for Drive and fast to load, but still larger than any input size you would train at (224–448).

## 1. Connect Google Drive and get the code

In [ ]:
import sys
assert "google.colab" in sys.modules, "This notebook is meant to run on Google Colab."

import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO = Path("/content/chest_xray_classifier")
DRIVE_DIR = Path("/content/drive/MyDrive/chest_xray_classifier")
STAGING = Path("/content/staging/images")   # Colab's local disk: fast, but wiped when the session ends
OUT_TAR = DRIVE_DIR / "images_512.tar"
SIZE = 512

DRIVE_DIR.mkdir(parents=True, exist_ok=True)
STAGING.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/vutr2/chest_xray_classifier.git", str(REPO)], check=True)

## 2. Which images are needed

This step keeps only the X-rays in the saved splits, so the train, validation and test patients match the results reported in the README.

In [ ]:
import pandas as pd

splits = pd.concat(pd.read_csv(REPO / "data" / "splits" / f"{s}.csv") for s in ["train", "val", "test"])
NEEDED = set(splits["Image Index"])
print(f"{len(NEEDED)} images needed")

## 3. Download, filter and shrink, one archive at a time

Each archive (about 4 GB) is processed like this:
1. The needed X-rays are read straight out of the archive.
2. They are resized on all CPU cores and saved.
3. The archive is deleted, so the Colab disk never holds more than one at a time.

If you re-run the cell in the same session, archives that are already done are skipped.

In [ ]:
import io
import tarfile
import time
from itertools import islice
from multiprocessing import Pool

from PIL import Image

PART_URLS = [
    "https://nihcc.box.com/shared/static/vfk49d74nhbxq3nqjg0900w5nvkorp5c.gz",
    "https://nihcc.box.com/shared/static/i28rlmbvmfjbl8p2n3ril0pptcmcu9d1.gz",
    "https://nihcc.box.com/shared/static/f1t00wrtdk94satdfb9olcolqx20z2jp.gz",
    "https://nihcc.box.com/shared/static/0aowwzs5lhjrceb3qp67ahp0rd1l1etg.gz",
    "https://nihcc.box.com/shared/static/v5e3goj22zr6h8tzualxfsqlqaygfbsn.gz",
    "https://nihcc.box.com/shared/static/asi7ikud9jwnkrnkj99jnpfkjdes7l6l.gz",
    "https://nihcc.box.com/shared/static/jn1b4mw4n6lnh74ovmcjb8y48h8xj07n.gz",
    "https://nihcc.box.com/shared/static/tvpxmn7qyrgl0w8wfh9kqfjskv6nmm1j.gz",
    "https://nihcc.box.com/shared/static/upyy3ml7qdumlgk2rfcvlb9k6gvqq2pj.gz",
    "https://nihcc.box.com/shared/static/l6nilvfa9cg3s28tqv1qc1olm3gnz54p.gz",
    "https://nihcc.box.com/shared/static/hhq8fkdgvcari67vfhs7ppg2w6ni4jze.gz",
    "https://nihcc.box.com/shared/static/ioqwiy20ihqwyr8pf4c24eazhh281pbu.gz",
]


def shrink(item):
    name, data = item
    Image.open(io.BytesIO(data)).convert("L").resize((SIZE, SIZE), Image.LANCZOS).save(STAGING / name)
    return name


def needed_images(archive):
    """Yield (file name, PNG bytes) for the needed X-rays, reading the archive front to back."""
    with tarfile.open(archive) as tar:
        for member in tar:
            name = Path(member.name).name
            if member.isfile() and name in NEEDED:
                yield name, tar.extractfile(member).read()


def batches(items, size=256):   # keeps memory use small: at most `size` images in flight
    items = iter(items)
    while batch := list(islice(items, size)):
        yield batch


for part, url in enumerate(PART_URLS, start=1):
    done = STAGING.parent / f".part{part:02d}.done"
    if done.exists():
        print(f"Part {part:02d} already done, skipping")
        continue
    start = time.time()
    archive = Path(f"/content/images_{part:02d}.tar.gz")
    subprocess.run(["curl", "-sSL", "--retry", "3", "-o", str(archive), url], check=True)
    with Pool() as pool:
        n = sum(len(pool.map(shrink, batch)) for batch in batches(needed_images(archive)))
    archive.unlink()
    done.touch()
    total = sum(1 for _ in STAGING.glob("*.png"))
    print(f"Part {part:02d}: kept {n} images in {time.time() - start:.0f}s | {total} of {len(NEEDED)} so far")

## 4. Check and save to Google Drive

In [ ]:
have = {p.name for p in STAGING.glob("*.png")}
missing = NEEDED - have
assert not missing, f"{len(missing)} images missing, e.g. {sorted(missing)[:3]}"
print(f"All {len(have)} images ready. Writing {OUT_TAR} ...")

subprocess.run(["tar", "-cf", str(OUT_TAR), "-C", str(STAGING.parent), "images"], check=True)
print(f"{OUT_TAR.stat().st_size / 1e9:.1f} GB written, waiting for Google Drive to finish uploading...")
drive.flush_and_unmount()
print("Done: MyDrive/chest_xray_classifier/images_512.tar")

## Next

Open `02_train.ipynb` in Colab and pick a GPU (**Runtime → Change runtime type → T4 GPU**). Its first cell connects Drive and unpacks this file.